# TRIBE v2 → short-video engagement (Kaggle)

Runs [neuro-engagement-forecasting](https://github.com/RehanKrish21/Neuro-engagement-forecasts) on free Kaggle GPUs. Meta's TRIBE v2 predicts brain responses to each video, and we test whether they forecast real engagement on [SnapUGC](https://github.com/dasongli1/SnapUGC_Engagement) better than simple baselines.

### One-time setup (right-hand panel)
1. **Session options → Accelerator → GPU T4 x2.** Needs a phone-verified account; re-check after importing, since Kaggle resets it.
2. **Session options → Internet → On.**
3. **Add-ons → Secrets → `HF_TOKEN`** = a Hugging Face **Read** token with access to [Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B), ticked for this notebook.

### Continuing from an earlier run (recommended)
Each run adds new videos to the ones already processed, so the dataset grows run after run. Before **Save & Run All**: **Add Input → Your Work → Notebooks →** pick this notebook (its latest version's output). Step 4 copies those predictions in and only processes new videos up to `N`.

### How to run
**Save Version → Save & Run All (Commit)** and close the tab. It runs everything in order (up to 12 h) and stops itself in time. Results land in the version's **Output** tab, and **Logs** shows progress. Running cells interactively works too.

Design note: every heavy step (TRIBE, torch) runs as a separate `neuroengage …` command in a fresh Python process. That makes the notebook immune to whatever Kaggle pre-imports, and no restart is ever needed.

*TRIBE v2 is CC BY-NC 4.0 (non-commercial). Check SnapUGC's terms too.*

## 1. Setup
Installs TRIBE v2, this package and their pinned dependencies, then verifies them **in a fresh process**.

In [ ]:
import importlib.metadata as md, os, shutil, subprocess, sys, time
from pathlib import Path
NOTEBOOK_START = time.time()

REPO = "https://github.com/RehanKrish21/Neuro-engagement-forecasts.git"
PKGS = [
    "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git",
    f"neuroengage @ git+{REPO}",
    "pyarrow", "gdown", "accelerate",
    # Preinstalled torchaudio is built for a newer torch; TRIBE pins torch 2.6, and a
    # mismatched torchaudio crashes transformers on import ("undefined symbol").
    "torchaudio==2.6.0",
]
MIN_NEUROENGAGE = (0, 9, 0)

def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

def at_least(v, minimum):
    return v is not None and tuple(int(x) for x in v.split(".")[:3]) >= minimum

def run(cmd, log=None, check=True):
    """Run a command, streaming its output here (and to `log` if given)."""
    out = open(log, "w") if log else None
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
        if out:
            out.write(line)
    code = proc.wait()
    if out:
        out.close()
    if check and code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {' '.join(map(str, cmd[:4]))} ... see the lines above")
    return code

NEUROENGAGE = [sys.executable, "-m", "neuroengage.cli"]

if not shutil.which("nvidia-smi"):
    raise SystemExit("❌ No GPU attached. Right-hand panel → Session options → Accelerator → GPU T4 x2, "
                     "then run this cell again.")
if sys.version_info < (3, 11):
    raise SystemExit(f"TRIBE v2 needs Python 3.11+, this image has {sys.version.split()[0]}.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
if not shutil.which("ffmpeg"):
    run(["bash", "-c", "apt-get update -qq && apt-get install -y -qq ffmpeg"])

ready = (version("tribev2") and at_least(version("neuroengage"), MIN_NEUROENGAGE) and version("accelerate")
         and (version("torch") or "").startswith("2.6") and (version("torchaudio") or "").startswith("2.6"))
if not ready:
    extra = ["--refresh-package", "neuroengage", "--reinstall-package", "neuroengage"]
    if run(["uv", "pip", "install", "--python", sys.executable, *extra, *PKGS], check=False) != 0:
        print("\nuv failed, retrying with pip ...\n")
        run([sys.executable, "-m", "pip", "install", *PKGS])

# Verify in a FRESH process: the notebook itself never imports torch or TRIBE.
run([sys.executable, "-c",
     "import torch, numpy, tribev2, neuroengage, transformers.models.llama.modeling_llama; "
     "from neuroengage.tribe_runner import TribeExtractor; "
     "print(f'✅ Ready: torch {torch.__version__}, numpy {numpy.__version__}, '"
     "      f'GPUs {torch.cuda.device_count()}, neuroengage {neuroengage.__version__}')"])
print("uvx:", bool(shutil.which("uvx")), "| ffmpeg:", bool(shutil.which("ffmpeg")))

## 2. Settings and Hugging Face login
Checks Llama 3.2 access up front, so a permissions problem shows up here and not 20 minutes into a run.

In [ ]:
SPLIT = "train"        # SnapUGC split: "train" or "val"
N = 700                # total videos to aim for, including earlier runs (the time budget decides how many finish)
SEED = 0
TIME_BUDGET_H = 11.0   # stop starting new videos before Kaggle's 12 h limit
PRECISION = "fp32"     # step 5b switches to "fp16" if it matches fp32 closely and is faster
RUN_PRECISION_CHECK = True
RUN_SMOKE_TEST = True  # step 3; set False once you know TRIBE works
MIRROR = "tcm03"       # fall back to an unofficial Hugging Face copy of SnapUGC's videos for dead links ("" = off)

OUT = Path("/kaggle/working/runs")      # saved with the notebook version (20 GB limit)
DATA = Path("/tmp/snapugc")             # temporary disk, not saved
CACHE = Path("/tmp/tribe_cache")
RUN = OUT / f"snapugc_{SPLIT}_seed{SEED}"   # same folder every run, so results accumulate
PREDS = RUN / "work" / "preds"
for p in (OUT, DATA, CACHE, PREDS):
    p.mkdir(parents=True, exist_ok=True)

from huggingface_hub import login, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))   # saved to disk, so subprocesses see it too
except Exception as exc:
    raise RuntimeError("Add your Hugging Face token: Add-ons → Secrets → HF_TOKEN, ticked for this notebook.") from exc
try:
    hf_hub_download("meta-llama/Llama-3.2-3B", "config.json")
    print("✅ Hugging Face login OK, Llama 3.2 access OK")
except Exception as exc:
    raise RuntimeError("Can't download Llama 3.2: access not approved yet, or a fine-grained token without "
                       "'read access to public gated repos'. Use a Read token.") from exc

## 3. Smoke test: one video
Runs TRIBE on a 52-second trailer. The first time, it downloads several GB of models (~10–20 min). Skip it with `RUN_SMOKE_TEST = False`.

In [ ]:
import urllib.request
import pandas as pd

if RUN_SMOKE_TEST:
    sample = Path("/tmp/sample.mp4")
    if not sample.exists():
        urllib.request.urlretrieve("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", sample)
    smoke_dir = OUT / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"video_id": ["sintel"], "path": [str(sample)]}).to_csv(smoke_dir / "manifest.csv", index=False)
    t0 = time.time()
    run([*NEUROENGAGE, "extract", "--manifest", smoke_dir / "manifest.csv", "--out", smoke_dir,
         "--cache", CACHE / "smoke", "--overwrite"])
    from neuroengage.tribe_runner import load_table
    smoke = load_table(smoke_dir / "network_timeseries")
    print(f"✅ TRIBE v2 works: {len(smoke)} predicted seconds (trailer is 52 s) in {time.time() - t0:.0f} s")
    display(smoke.head())
else:
    print("Skipped")

## 4. Get SnapUGC
SnapUGC's download is a **label file** (`train_data.csv`: `Id, Title, Description, Download_link, ECR`, ~106k rows). This step fetches it, then downloads **only a stratified sample** of videos from their `Download_link`s (spread across the engagement range, with 30% extra in case some links are dead). Re-running skips finished downloads; failures are logged in `_download_log.csv`. About two-thirds of the official Snapchat links are dead, so with `MIRROR = "tcm03"` those videos are fetched from an unofficial Hugging Face copy instead (matched by video ID, ~121k videos; status `ok_mirror`). Labels always come from the official CSV.

The train split has **ECR** (share of viewers who stay ≥ 5 s) but no NAWP, so the analysis uses ECR.

In [ ]:
import gdown

LOCAL_LABELS = None   # path to train_data.csv if you uploaded it yourself (e.g. under /kaggle/input/...)
FILE_IDS = {"train": "1Mv5Esq5gGuxRTayabRUb5NmHwEN3JdbD", "val": "1iY7qUfqivYzUqUID501dKQDpI3NQil7y"}
dl_dir = DATA / "download"
dl_dir.mkdir(parents=True, exist_ok=True)
VIDEOS = DATA / f"{SPLIT}_videos"

if LOCAL_LABELS:
    LABELS = Path(LOCAL_LABELS)
else:
    found = sorted(dl_dir.glob("*.csv"))
    LABELS = found[0] if found else Path(gdown.download(id=FILE_IDS[SPLIT], output=f"{dl_dir}/", quiet=False))
print("Labels:", LABELS, f"({sum(1 for _ in open(LABELS)) - 1:,} rows)")

# Carry over predictions from earlier runs attached as inputs (Add Input → Your Work → this notebook)
import glob
copied = 0
for f in glob.glob("/kaggle/input/**/preds/*.npz", recursive=True):
    dest = PREDS / Path(f).name
    if not dest.exists():
        shutil.copy2(f, dest)
        copied += 1
print(f"Carried over {copied} predictions from earlier runs; {len(list(PREDS.glob('*.npz')))} available in total")

run([*NEUROENGAGE, "snapugc-download", "--labels", LABELS, "--videos-dir", VIDEOS,
     "--n", N, "--seed", SEED, "--workers", 8, "--done-dir", PREDS, *(["--mirror", MIRROR] if MIRROR else [])])
n_videos = len(list(VIDEOS.glob("*.mp4"))) if VIDEOS.exists() else 0
if n_videos == 0 and not any(PREDS.glob("*.npz")):
    raise RuntimeError("No videos downloaded: the CDN links may have expired. Send the status counts above.")
print(f"✅ {n_videos} videos in {VIDEOS}")

## 5. Run TRIBE v2 on the downloaded videos
**5a** picks up to `N` videos spread across the engagement range. **5b** (~10 min) checks whether half precision (fp16) gives the same brain predictions as full precision on the 2 shortest videos, and adopts it only if every network agrees at r ≥ 0.98 and it's faster. **5c** runs TRIBE on every GPU at once (T4 x2 = 2 videos in parallel), printing progress and time left. Re-running resumes; finished videos are never redone.

In [ ]:
manifest_path = RUN / "manifest.csv"
# Rebuilt every session: keeps every video that already has predictions, then adds new ones up to N.
run([*NEUROENGAGE, "snapugc-prepare", "--labels", LABELS, "--videos-dir", VIDEOS,
     "--n", N, "--seed", SEED, "--out", manifest_path, "--done-dir", PREDS])
manifest = pd.read_csv(manifest_path, dtype={"video_id": str})
GPUS = [int(g) for g in subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                                       capture_output=True, text=True).stdout.split()]
done = sum((RUN / "work" / "preds" / f"{v}.npz").exists() for v in manifest.video_id)
print(f"{len(manifest)} videos in this run ({done} already done); GPUs: {GPUS}")

### 5b. Half-precision check

In [ ]:
import numpy as np
from neuroengage.tribe_runner import load_table

if RUN_PRECISION_CHECK and PRECISION == "fp32":
    on_disk = manifest[[Path(p).is_file() for p in manifest["path"]]]
    sizes = [Path(p).stat().st_size for p in on_disk["path"]]
    sample = on_disk.assign(_size=sizes).nsmallest(2, "_size").drop(columns="_size")
    check_dir = RUN / "precision_check"
    check_dir.mkdir(exist_ok=True)
    sample.to_csv(check_dir / "manifest.csv", index=False)
    tables, secs = {}, {}
    for prec in ("fp32", "fp16"):
        t0 = time.time()
        code = run([*NEUROENGAGE, "extract", "--manifest", check_dir / "manifest.csv", "--out", check_dir / prec,
                    "--cache", CACHE / f"check_{prec}", "--precision", prec, "--overwrite"],
                   log=check_dir / f"{prec}.log", check=False)
        secs[prec] = time.time() - t0
        if code == 0:
            tables[prec] = load_table(check_dir / prec / "network_timeseries")
    if set(tables) == {"fp32", "fp16"}:
        m = tables["fp32"].merge(tables["fp16"], on=["video_id", "t"], suffixes=("_32", "_16"))
        cols = [c for c in tables["fp32"].columns if c not in ("video_id", "t", "pattern_shift")]
        r = pd.Series({c: np.corrcoef(m[c + "_32"], m[c + "_16"])[0, 1] for c in cols}).round(4)
        speedup = secs["fp32"] / secs["fp16"]
        print(r.to_string())
        print(f"\nfp32 {secs['fp32']:.0f} s vs fp16 {secs['fp16']:.0f} s for 2 videos → {speedup:.1f}x faster; "
              f"lowest agreement r = {r.min():.4f}")
        if r.min() >= 0.98 and speedup >= 1.3:
            PRECISION = "fp16"
            print("✅ fp16 matches fp32 closely and is faster: using PRECISION = 'fp16'")
        else:
            print("Keeping PRECISION = 'fp32' (fp16 not close enough or not faster)")
    else:
        print(f"Keeping PRECISION = 'fp32' (a check run failed; logs in {check_dir})")
else:
    print(f"Skipped; PRECISION = {PRECISION!r}")

### 5c. Extract on all GPUs

In [ ]:
from neuroengage.parallel import run_extract_on_gpus

hours_left = TIME_BUDGET_H - (time.time() - NOTEBOOK_START) / 3600
print(f"GPUs {GPUS}, precision {PRECISION}, {hours_left:.1f} h of budget left")
ts_path = run_extract_on_gpus(manifest_path, RUN / "work", GPUS, CACHE / PRECISION, precision=PRECISION,
                              max_hours=max(hours_left - 0.4, 0.1), poll_seconds=120)
timeseries = load_table(ts_path)
print(f"✅ {timeseries['video_id'].nunique()}/{len(manifest)} videos ready for step 6")

## 6. Brain features vs baselines
Pre-registered analysis ([docs/preregistration.md](https://github.com/RehanKrish21/Neuro-engagement-forecasts/blob/main/docs/preregistration.md)). **Primary comparison:** a fixed 50/50 rank average of the baseline and brain models vs the baseline alone, on ECR (share of viewers who stay past 5 s). Reports pooled held-out Spearman and pairwise accuracy (chance = 0.5) with 95% bootstrap intervals, plus the paired gain over baseline. Secondary: brain-only, a compact brain set (mean + hook only), and single combined models.

Also reports an **exploratory raw-feature baseline**: TRIBE's own inputs (V-JEPA 2, Wav2Vec-BERT, Llama 3.2 features), recorded during extraction at no extra GPU cost. It asks whether the brain step adds anything beyond them. Only videos processed with v0.8.0+ have raw features.

In [ ]:
from IPython.display import Image, Markdown, display

MIN_VIDEOS = 50   # below this the comparison is noise
done_ids = set(timeseries["video_id"])
if len(done_ids) < MIN_VIDEOS:
    print(f"Only {len(done_ids)} videos finished; need at least {MIN_VIDEOS}. Run step 5c again (it resumes).")
else:
    run([*NEUROENGAGE, "train-scores", "--manifest", manifest_path, "--timeseries", ts_path,
         "--targets", "nawp,ecr", "--baseline-cols", "duration_s,title_n_words,has_description",
         "--hook-seconds", 5, "--seed", SEED, "--n-boot", 2000, "--out", RUN / "results",
         # run 1's videos were used to choose the analysis: also test on the other videos only
         *(["--exploratory-ids", "snapugc-run1"] if SPLIT == "train" else []),
         # exploratory: TRIBE's own input features as a baseline (saved since v0.8.0, no extra GPU time)
         "--raw-from", PREDS,
         "--title", f"SnapUGC ({len(done_ids)} videos): TRIBE v2 brain features vs baselines"])
    display(Image(str(RUN / "results" / "summary.png")))
    if (RUN / "results" / "raw_comparison.png").exists():
        display(Image(str(RUN / "results" / "raw_comparison.png")))
    display(Markdown((RUN / "results" / "report.md").read_text()))

**Reading it:** look at the primary row's **gain 95% CI**. If it's entirely above 0, brain features add predictive value beyond the baseline at this sample size. If it straddles 0, the data can't tell yet: add videos (run again with this output attached). `pairwise_acc` is the share of video pairs ordered correctly: 0.5 is chance.

## 7. Package results
Everything under `/kaggle/working` is saved with the notebook version (**Output** tab). This also writes a single zip.

In [ ]:
if (RUN / "results").exists():
    zip_path = shutil.make_archive(f"/kaggle/working/{RUN.name}_results", "zip", RUN / "results")
    print(zip_path, f"{Path(zip_path).stat().st_size / 1e6:.1f} MB")
else:
    print("No results yet (step 6 was skipped); predictions so far are saved under", RUN / "work")
print(shutil.disk_usage("/kaggle/working"))